# 06: Generators and Readouts

## Objective
Learn to use waveform generators and optimize readout resonators.

In [ ]:
BITSTREAM_PATH = "/path/to/firmware.bit"
USE_PROXY = False
PROXY_IP = "192.168.1.100"

from qick import QickSoc, QickConfig
import numpy as np
import matplotlib.pyplot as plt

if USE_PROXY:
    from qick.pyro import make_proxy
    soc = make_proxy(PROXY_IP)
else:
    soc = QickSoc(bitfile=BITSTREAM_PATH)

soc.reset_gens()
soc.reset_adcs()
print("Ready")

In [ ]:
config = QickConfig(reps=1000, soft_avgs=10, hard_avgs=1)

config.gen_channel(ch=0, freq=100e6, phase=0, gain=0.5, style="const")

soc.run(config)
print("Generator running")

In [ ]:
config = QickConfig(reps=1, soft_avgs=100)

config.readout_channel(ch=0, freq=150e6, gain=0.3, length=1024)
config.pulse(ch=0, style="gaussian", length=200, gain=0.5, phase=0)

soc.run(config)
i, q = soc.get_readout()

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(i, label='I')
plt.plot(q, label='Q')
plt.legend()
plt.title("Readout IQ")
plt.subplot(1,2,2)
plt.plot(i + 1j*q, '.')
plt.axis('equal')
plt.title("Constellation")
plt.show()

In [ ]:
freqs = np.linspace(140e6, 160e6, 21)
results = []

for f in freqs:
    config = QickConfig(reps=1, soft_avgs=100)
    config.readout_channel(ch=0, freq=f, gain=0.3, length=1024)
    config.pulse(ch=0, style="gaussian", length=200, gain=0.5)
    soc.run(config)
    i, q = soc.get_readout()
    results.append(np.mean(i + 1j*q))

plt.figure()
plt.plot(freqs/1e6, np.abs(results))
plt.xlabel("Frequency (MHz)")
plt.ylabel("|IQ| (a.u.)")
plt.title("Readout Resonator Response")
plt.grid(True)
plt.show()